<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, FileUpload, Output
from IPython.display import display, Markdown, HTML, clear_output
import io
import base64

# === 1. Oppsett ===
uploader = FileUpload(accept='', multiple=False, description="Last opp måledata")
app_display = Output()

def start_analysen(change):
    with app_display:
        clear_output(wait=True)
        if not uploader.value: return

        # --- SKUDDSIKKER FILHENTING ---
        try:
            raw = uploader.value
            if isinstance(raw, (list, tuple)):
                file_info = raw[0]
            elif isinstance(raw, dict):
                file_info = list(raw.values())[0]
            else:
                file_info = raw

            content = file_info['content']
            df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')
            tid_data, niva_data = df.iloc[:,0].values, df.iloc[:,1].values
        except Exception as e:
            print(f"Feil ved lesing av fil: {e}")
            return

        # === 2. ESTIMERING (10/85-regel) ===
        y0_v = niva_data[0]
        A_v = niva_data[-1] - y0_v
        t10 = tid_data[np.where(niva_data > y0_v + 0.10 * A_v)[0][0]] if len(np.where(niva_data > y0_v + 0.10 * A_v)[0]) > 0 else tid_data[0]
        t85 = tid_data[np.where(niva_data > y0_v + 0.85 * A_v)[0][0]] if len(np.where(niva_data > y0_v + 0.85 * A_v)[0]) > 0 else tid_data[-1]
        t63 = tid_data[np.where(niva_data > y0_v + 0.63 * A_v)[0][0]] if len(np.where(niva_data > y0_v + 0.63 * A_v)[0]) > 0 else tid_data[-1]

        L_est = max(0, float(t10 - 0.05 * (t85 - t10)))
        T_est = max(0.1, float(t63 - L_est))

        # === 3. PLOTT-FUNKSJON MED ALLE LINJER ===
        plot_out = Output(layout={'width': 'auto', 'max_width': '650px'})

        def update_plot(change=None):
            with plot_out:
                clear_output(wait=True)
                A, T, L, y0 = A_s.value, T_s.value, L_s.value, y0_s.value
                y_model = np.where(tid_data < L, y0, y0 + A * (1 - np.exp(-(tid_data - L) / T)))

                fig, ax = plt.subplots(figsize=(8, 5))
                ax.plot(tid_data, niva_data, "b.", markersize=3, alpha=0.3, label="Måledata")
                ax.plot(tid_data, y_model, "r-", linewidth=2, label="Modell")

                # y0 linje
                ax.axhline(y0, color='black', linestyle='--', alpha=0.4)
                ax.text(tid_data[0]+10, y0, f' y0={y0:.1f}', fontweight='bold', va='bottom')

                # Dødtid L (oransje)
                ax.axvline(L, color='orange', linestyle=':', linewidth=2)
                ax.text(L+110, y0+30, f' L={L:.1f}s', color='orange', fontweight='bold', ha='right')

                # 63% respons og T (grønn)
                y63 = y0 + 0.63 * A
                t63_target = L + T
                ax.axhline(y63, color='green', linestyle=':', alpha=0.5)
                ax.axvline(t63_target, color='green', linestyle=':', alpha=0.5)
                ax.plot(t63_target, y63, 'go', markersize=8)
                ax.text(t63_target, y63, f' y63={y63:.1f}\n T={T:.1f}s', color='green', fontweight='bold', va='top', ha='left')

                # Gain dy (lilla)
                ax.vlines(tid_data[-1], y0, y0+A, color='purple', linewidth=3)
                ax.text(tid_data[-1], y0+A/2, f' Δy={A:.1f}', color='purple', fontweight='bold', ha='left')

                ax.grid(True, alpha=0.3)
                ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå")
                ax.legend(loc='lower right', fontsize='small')
                plt.tight_layout()
                plt.show()

        # === 4. WIDGETS ===
        style = {'description_width': '40px'}
        s_layout = {'width': '280px', 'min_width': '280px'}

        A_s = FloatSlider(value=A_v, min=min(0, A_v*0.2), max=A_v*2, step=0.01, description="Δy", style=style, layout=s_layout)
        T_s = FloatSlider(value=T_est, min=0.1, max=T_est*4, step=0.1, description="T", style=style, layout=s_layout)
        L_s = FloatSlider(value=L_est, min=0, max=tid_data[-1]/2, step=0.1, description="L", style=style, layout=s_layout)
        y0_s = FloatSlider(value=y0_v, min=y0_v-10, max=y0_v+10, step=0.01, description="y0", style=style, layout=s_layout)

        for s in [A_s, T_s, L_s, y0_s]: s.observe(update_plot, "value")

        # === 5. KOMPLETT TABELL (3 KOLONNER) ===
        info_html = r"""
        <div style="font-family: sans-serif; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background: #fafafa; min-width: 350px; max-width: 500px;">
            <h4 style="margin-top:0; text-align: center;">SIMC Regulering</h4>
            <table style="width: 100%; border-collapse: collapse; background: white; font-size: 0.85em;">
                <tr style="background: #eee;">
                    <th style="padding: 5px; border: 1px solid #ccc;">λ</th>
                    <th style="padding: 5px; border: 1px solid #ccc;">Respons</th>
                    <th style="padding: 5px; border: 1px solid #ccc;">Observasjon</th>
                </tr>
                <tr><td style="padding: 5px; border: 1px solid #ccc; text-align: center;">T / 2</td><td style="padding: 5px; border: 1px solid #ccc;">Rolig</td><td style="padding: 5px; border: 1px solid #ccc;">Lite oversving</td></tr>
                <tr><td style="padding: 5px; border: 1px solid #ccc; text-align: center;">T / 4</td><td style="padding: 5px; border: 1px solid #ccc;">Standard</td><td style="padding: 5px; border: 1px solid #ccc;">God balanse</td></tr>
                <tr><td style="padding: 5px; border: 1px solid #ccc; text-align: center;">T / 6</td><td style="padding: 5px; border: 1px solid #ccc;">Rask</td><td style="padding: 5px; border: 1px solid #ccc;">Aggressiv</td></tr>
            </table>
            <div style="background: #fff; padding: 10px; border-radius: 5px; border: 1px solid #eee; margin-top: 10px; font-size: 0.9em;">
                <b>K = Δy / Δu</b><br>
                <b>Kp = T / (K · (λ + L))</b><br>
                <b>Ti = min(T, 4 · (λ + L))</b>
            </div>
        </div>
        """
        info_widget = widgets.HTML(value=info_html, layout={'margin': '0 0 0 40px'})

        kontroller = VBox([A_s, T_s, L_s, y0_s], layout={'width': '300px', 'min_width': '300px'})
        dashbord = HBox([plot_out, kontroller, info_widget], layout={'flex_flow': 'row wrap', 'align_items': 'flex-start'})

        display(Markdown(f"### Modell-identifikasjon fullført"), dashbord)
        update_plot()

# === Start ===
uploader.observe(start_analysen, names='value')
display(Markdown("# FOPDT Simulator"), uploader, app_display)